# Problem 56: Agentic RAG

**Difficulty:** Hard | **Framework:** OpenAI Agents SDK

**Categories:** rag, orchestration

This notebook accompanies [Agent Foundry](https://agent-foundry-theta.vercel.app/problems/agentic-rag).

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The agent must decide whether retrieval is needed for each query
- The agent must formulate its own search queries
- The agent must evaluate retrieved results and decide if more retrieval is needed
- Simple factual questions the agent knows should be answered directly without retrieval


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from agents import Agent, Runner
from agents.tool import function_tool
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings()

documents = [
    Document(page_content="Our enterprise plan includes SSO, audit logs, and 99.9% SLA."),
    Document(page_content="The API rate limit for enterprise customers is 10,000 requests per minute."),
    Document(page_content="Data residency options: US-East, EU-West, and AP-Southeast regions."),
    Document(page_content="Custom integrations require a minimum 12-month contract."),
    Document(page_content="Our SOC 2 Type II certification was renewed in January 2024."),
]

vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever()

@function_tool
def search_docs(query: str) -> str:
    """Search the knowledge base for relevant documents."""
    docs = retriever.invoke(query)
    return "\n".join([doc.page_content for doc in docs])

# BUG: Always retrieves regardless of query type — no decision logic
agent = Agent(
    name="Knowledge Assistant",
    instructions="You answer questions by searching the knowledge base. Always use the search tool.",
    tools=[search_docs],
    model="gpt-4o-mini",
)

for q in ["What is the enterprise API rate limit?", "What is 2 + 2?"]:
    result = Runner.run_sync(agent, q)
    print(result.final_output)

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Add a routing step where the agent classifies the query as needing retrieval or direct answer
2. After retrieval, add an evaluation step where the agent checks if the results are sufficient
3. Use a loop — if the first retrieval is insufficient, the agent reformulates and retrieves again


## 7. Evaluation Criteria

Check your solution against these criteria:

- Decides when to retrieve
- Formulates search queries
- Evaluates retrieval sufficiency
- Routes queries correctly
